# F03 SIGISMUND — Réacteur Remotion SVG Neon
## PENTERACT DORN V3 — VIIe Légion

**Rôle** : Rendu Remotion 60fps, courbes SVG neon, timing dynamique, caméra virtuelle.

**Entrées** : `F03_SIGISMUND/IN/plan_de_vol.json` (validé par F02) + `IN/*.png`  
**Sorties** : `F03_SIGISMUND/OUT/video_render.mp4`

---
### Avant de lancer
1. Monte ton Google Drive (cellule 1)
2. Choisis le mode de rendu quand demandé (cellule 2)
3. Si mode `github` : colle ton PAT quand la cellule 3 te le demande
4. Lance les cellules suivantes dans l'ordre

### Mode github — prérequis
- `GITHUB_TOKEN` : PAT avec permissions `repo` + `packages:write`
- Le secret `GH_TOKEN` est **configuré automatiquement** dans le repo (cellule 3)


In [ ]:
# CELLULE 1 — Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Drive monté.')

In [ ]:
# CELLULE 2 — Configuration générale
DRIVE_BASE = '/content/drive/MyDrive/DRIVE_DORN'  # ← adapter si besoin
REPO       = 'kioka8877-ux/DORN'

print('Mode de rendu :')
print('  1 — direct  (1 worker Colab, lent, aucune config)')
print('  2 — modal   (3 workers Modal, nécessite compte Modal)')
print('  3 — github  (10 workers GitHub Actions, recommandé)')
print()

_choix = input('Votre choix [1/2/3] : ').strip()
_modes = {'1': 'direct', '2': 'modal', '3': 'github'}
if _choix not in _modes:
    raise ValueError(f'Choix invalide : {_choix!r} — entrez 1, 2 ou 3')
MODE = _modes[_choix]

print(f'\nDrive base : {DRIVE_BASE}')
print(f'Mode rendu : {MODE}')
print(f'Repo       : {REPO}')
if MODE == 'github':
    print('[INFO] Lance la cellule 3 pour configurer le token GitHub.')


In [ ]:
# CELLULE 3 — Authentification GitHub (MODE = 'github' uniquement)
# Le token est saisi de façon sécurisée, validé, puis configuré
# automatiquement comme secret GH_TOKEN dans le repo DORN.
import os, getpass, base64, requests

if MODE != 'github':
    print(f'[SKIP] Mode actuel : {MODE} — cellule non requise.')
else:
    import nacl.public, nacl.encoding

    # ── 1. Saisie sécurisée ───────────────────────────────────────────────
    token = getpass.getpass('Collez votre GitHub PAT puis appuyez sur Entrée : ')
    token = token.strip()
    os.environ['GITHUB_TOKEN'] = token
    print('[OK] Token chargé en mémoire de session')

    # ── 2. Validation du token ────────────────────────────────────────────
    headers = {'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json'}
    r = requests.get(f'https://api.github.com/repos/{REPO}', headers=headers)
    if r.status_code != 200:
        raise RuntimeError(f'Token invalide ou permissions insuffisantes (HTTP {r.status_code})')
    print(f'[OK] Accès repo vérifié : {REPO}')

    # ── 3. Récupération de la clé publique du repo ────────────────────────
    r_key = requests.get(
        f'https://api.github.com/repos/{REPO}/actions/secrets/public-key',
        headers=headers
    )
    r_key.raise_for_status()
    pub_key_data = r_key.json()
    pub_key_b64  = pub_key_data['key']
    key_id       = pub_key_data['key_id']

    # ── 4. Chiffrement (libsodium sealed box) ─────────────────────────────
    pub_key_bytes = base64.b64decode(pub_key_b64)
    pk  = nacl.public.PublicKey(pub_key_bytes)
    box = nacl.public.SealedBox(pk)
    encrypted     = box.encrypt(token.encode('utf-8'))
    encrypted_b64 = base64.b64encode(encrypted).decode('utf-8')

    # ── 5. Push du secret GH_TOKEN dans le repo ───────────────────────────
    r_secret = requests.put(
        f'https://api.github.com/repos/{REPO}/actions/secrets/GH_TOKEN',
        headers=headers,
        json={'encrypted_value': encrypted_b64, 'key_id': key_id}
    )
    if r_secret.status_code in (201, 204):
        print('[OK] Secret GH_TOKEN configuré automatiquement dans le repo DORN')
    else:
        print(f'[WARN] Impossible de setter GH_TOKEN automatiquement (HTTP {r_secret.status_code})')
        print('       → Allez dans DORN > Settings > Secrets > Actions > New secret')
        print('       → Nom : GH_TOKEN | Valeur : votre token')

    print(f'\n[PRÊT] Token actif pour toute la session.')


In [ ]:
# CELLULE 4 — Installation dépendances (requests, pynacl, Node.js)
import subprocess, sys

# pip
subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests', 'pynacl', '-q'], check=True)
print('[OK] requests + pynacl installés')

# Node.js
result = subprocess.run(['node', '--version'], capture_output=True)
if result.returncode != 0:
    print('Installation Node.js 20.x ...')
    subprocess.run(['bash', '-c',
        'curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs'],
        check=True)
else:
    print(f'[OK] Node.js : {result.stdout.decode().strip()}')

r2 = subprocess.run(['npm', '--version'], capture_output=True)
print(f'[OK] npm : {r2.stdout.decode().strip()}')

In [ ]:
# CELLULE 5 — Copie du codebase depuis Drive + npm install
import shutil, os
from pathlib import Path

codebase_src = Path(DRIVE_BASE) / 'F03_SIGISMUND' / 'CODEBASE'
codebase_dst = Path('/content/F03_CODEBASE')

if codebase_dst.exists():
    shutil.rmtree(codebase_dst)
shutil.copytree(codebase_src, codebase_dst)
print(f'[OK] Codebase copié : {codebase_dst}')

script_dst = '/content/drn_f03_sigismund.py'
shutil.copy2(codebase_dst / 'drn_f03_sigismund.py', script_dst)
print(f'[OK] Script copié : {script_dst}')

print('npm install ...')
subprocess.run(['npm', 'install', '--prefer-offline'], cwd=str(codebase_dst), check=True)
print('[OK] npm install terminé')

In [ ]:
# CELLULE 6 — Lancement du rendu F03 SIGISMUND
import subprocess, sys, os

cmd = [
    sys.executable, script_dst,
    '--mode',       MODE,
    '--drive-base', DRIVE_BASE,
]

if MODE == 'github':
    github_token = os.environ.get('GITHUB_TOKEN', '')
    if not github_token:
        raise ValueError('[STOP] GITHUB_TOKEN manquant — lancer la cellule 3 d\'abord')
    cmd += ['--github-token', github_token, '--repo', REPO]

result = subprocess.run(cmd, capture_output=False)

if result.returncode == 0:
    print('\n[OK] F03 SIGISMUND — RENDU OK')
    print(f'→ Vidéo dans : {DRIVE_BASE}/F03_SIGISMUND/OUT/video_render.mp4')
    print('→ Étape suivante : F04A INWIT')
else:
    print('\n[FAIL] F03 SIGISMUND — RENDU FAIL — corriger les erreurs ci-dessus')


In [ ]:
# CELLULE 7 — Transit CRS_CUSTOS (check-in F03)
# Lancer UNIQUEMENT si la cellule 6 s'est terminée avec RENDU OK
import shutil, subprocess, sys, os
from pathlib import Path

custos_src = Path(DRIVE_BASE) / 'CRS_CUSTOS.py'
shutil.copy2(custos_src, '/content/CRS_CUSTOS.py')

result = subprocess.run(
    [sys.executable, '/content/CRS_CUSTOS.py',
     '--frigate', 'F03', '--mode', 'check-in', '--drive-base', DRIVE_BASE],
    capture_output=False
)
if result.returncode == 0:
    print('\n[OK] CRS_CUSTOS — F03 check-in OK — Transit autorisé vers F04')
else:
    print('\n[FAIL] CRS_CUSTOS — F03 check-in FAIL')